In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,  GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


In [2]:
df=pd.read_excel("../data/cleaned_data.xlsx")

In [3]:
df

,Property_ID,Area_sqft,Bedrooms,Bathrooms,Floors,Year_Built,Location_Score,School_Score,Current_Price,Price_Year_1,Price_Year_2,Price_Year_3,Price_Year_4,Price_After_5_Years,Price_Category,Price_per_sqft,Area_per_Bedroom,Property_age
0,1,2751,3,4,3,1989,54,58,624636,656833,690689,726290,763726,803092,Medium,227.057797,917.000000,37
1,2,3322,5,1,1,2019,62,78,694237,729166,765853,804385,844856,887363,Medium,208.981638,664.400000,7
2,3,4765,3,3,2,1989,57,50,771778,803688,836917,871520,907554,945078,Medium,161.968101,1588.333333,37
3,4,4512,2,2,3,2004,57,90,792597,854577,921404,993457,1071144,1154906,High,175.664229,2256.000000,22
4,5,2436,3,1,1,1987,76,69,580257,612034,645552,680905,718194,757525,Low,238.200739,812.000000,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,1563,5,1,2,2018,72,69,487618,516856,547847,580696,615515,652422,Low,311.975688,312.600000,8
496,497,4682,4,1,2,1982,69,96,934003,997429,1065162,1137495,1214740,1297231,High,199.488039,1170.500000,44
497,498,4205,3,4,1,1989,96,52,821328,855209,890488,927222,965472,1005300,Medium,195.321760,1401.666667,37
498,499,3413,5,2,1,1998,96,63,746551,806353,870945,940712,1016067,1097459,High,218.737474,682.600000,28


In [4]:
X= df.drop(columns=['Property_ID','Year_Built', 'Price_Year_1', 'Price_Year_2', 'Price_Year_3', 'Price_Year_4', 'Price_After_5_Years','Price_Category'], axis=1)
y= df['Price_After_5_Years']

In [5]:
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.2, random_state=42)

In [6]:
X.shape

(500, 10)

In [7]:
y.shape

(500,)

In [8]:
scaler= StandardScaler()
X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)

In [9]:
models={
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "XGBoost": XGBRegressor()
}

In [12]:
for name, model in models.items():

    if name == "Linear Regression" or name == "Ridge Regression":
        model.fit(X_train_scaled, y_train)
        y_pred= model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred= model.predict(X_test)
    mae= mean_absolute_error(y_test, y_pred)
    rmse= np.sqrt(mean_squared_error(y_test, y_pred))
    r2= r2_score(y_test, y_pred)
    print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.2f}")


Linear Regression: MAE=72770.18, RMSE=87092.38, R2=0.86
Ridge Regression: MAE=72137.42, RMSE=86560.49, R2=0.86
Decision Tree: MAE=100561.85, RMSE=122174.07, R2=0.72
Random Forest: MAE=81033.85, RMSE=99308.59, R2=0.81
Gradient Boosting: MAE=79577.11, RMSE=95479.79, R2=0.83
XGBoost: MAE=87435.90, RMSE=108389.65, R2=0.78


In [15]:
for name, model in models.items():
    if name == "Linear Regression" or name == "Ridge Regression":
        model.fit(X_train_scaled, y_train)
        y_pred= model.predict(X_train_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred= model.predict(X_train)
    
    mae= mean_absolute_error(y_train, y_pred)
    mse= mean_squared_error(y_train, y_pred)
    rmse= np.sqrt(mse)
    r2= r2_score(y_train, y_pred)
    
    print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.2f}")

Linear Regression: MAE=68788.77, RMSE=82973.93, R2=0.87
Ridge Regression: MAE=68788.97, RMSE=83018.64, R2=0.87
Decision Tree: MAE=0.00, RMSE=0.00, R2=1.00
Random Forest: MAE=25891.77, RMSE=32375.94, R2=0.98
Gradient Boosting: MAE=41055.32, RMSE=52102.36, R2=0.95
XGBoost: MAE=324.53, RMSE=481.06, R2=1.00


In [19]:
param_grid={
    "alpha": [0.01, 0.1, 0.5, 1, 2, 5, 10, 20, 50, 100]
}
grid_search= GridSearchCV(
    Ridge(),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)

In [20]:
grid_search.fit(X_train_scaled, y_train)
grid_search.best_params_

{'alpha': 0.01}

In [21]:
grid_search.best_estimator_

,alpha,0.01
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [ ]:
y_pred_ridge_tuned= grid_search.best_estimator_.predict(X_test_scaled)
mae= mean_absolute_error(y_test, y_pred_ridge_tuned)
rmse= np.sqrt(mean_squared_error(y_test, y_pred_ridge_tuned))
r2= r2_score(y_test, y_pred_ridge_tuned)
print(f"Mean absolute error: {mae:.2f} ,Root mean squared error: {rmse:.2f}, R2 score: {r2:.2f}")

Mean absolute error: 72762.92 ,Root mean squared error: 87085.94, R2 score: 0.86


In [30]:
final_model= Ridge(alpha=1)
final_model.fit(X_train_scaled, y_train)
y_pred_final= final_model.predict(X_test_scaled)

In [31]:
mae= mean_absolute_error(y_test, y_pred_final)
rmse= np.sqrt(mean_squared_error(y_test, y_pred_final))
r2= r2_score(y_test, y_pred_final)
print(f"Mean absolute error: {mae:.2f} ,Root mean squared error: {rmse:.2f}, R2 score: {r2:.2f}")

Mean absolute error: 72137.42 ,Root mean squared error: 86560.49, R2 score: 0.86


In [34]:
joblib.dump(final_model, "../models/ridge_model.pkl")

['../models/ridge_model.pkl']

In [35]:
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']